<a href="https://colab.research.google.com/github/pizzeman/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [90]:
import pandas as pd
PATH = '../product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [91]:
# TODO (hint: .str.lower().str.contains('pop'))
pop = cat[cat['product_name'].str.lower().str.contains('pop')]
print(f"Number of pop: {len(pop)}")
pop

Number of pop: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [92]:
# TODO
straw_pop = pop[pop['product_name'].str.lower().str.contains('strawberry')]
not_straw_pop = pop[~pop['product_name'].str.lower().str.contains('strawberry')]
print(f"Number of strawberry pop: {len(straw_pop)}")
print(straw_pop)
print()
print(f"Number of non-strawberry pop: {len(not_straw_pop)}")
not_straw_pop

Number of strawberry pop: 4
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  

Number of non-strawberry pop: 0


,sku,product_name,category,subcategory,unit_cost,supplier,package_size


**Pop-Tart rows that are a different product:** All of them are strawberry; rows 1-4

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [93]:
# TODO
CANON = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW'
}
cat['canonical'] = cat['sku'].map(CANON).fillna(cat['sku'])
cat

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,POPTART-STRAW
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack,WATER24PK
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack,WTR_24PK
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,AA,NaN,3M Company,6 pack,BATT-100
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk,BAT-AA4
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack,BATT_AA_4PK


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [94]:
print("BEFORE")
print(len(cat['category']))
cat['category'].value_counts()


BEFORE
45


,count
category,
Emergency,5
Emergency Supplies,3
EMERGENCY,3
ENTERTAINMENT,3
GROCERY,3
Breakfast,2
entertainment,2
household,2
ELECTRONICS,2


In [95]:
# TODO
print("AFTER")
cat['category'] = cat['category'].str.strip().str.title()
print(len(cat['category']))
cat['category'].value_counts()


AFTER
45


,count
category,
Emergency,9
Entertainment,5
Grocery,5
Electronics,4
Household,4
Emergency Supplies,3
Breakfast,3
Baby,3
Snacks,2


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [96]:
# TODO
category = cat.copy()
category['canonical'] = (category['product_name'].str.lower()
                         .str.replace(r'[^a-z ]', '', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (category.groupby('canonical')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates


,sku
canonical,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [97]:
# TODO
assert category['canonical'].notna().all()
assert len(category) == len(cat)
category

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,poptarts strawberry
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,poptart strawberry
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,poptarts strawberry
3,PT_STRAWBERRY,pop-tarts strawberry,Breakfast,Pop-Tarts,NaN,Campbell's,4 pack,poptarts strawberry
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,bottled pk water
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack,bottled pack water
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack,pack water
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,AA,NaN,3M Company,6 pack,aa batteries pk
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk,aa batteries pack
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack,aa batteries pack


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

Bottled water in a 24 pack is also fragmented. I grouped them based on those that were in a 24 pack. If it was a different size, then that would be a different product. Also, chip variety pack has different SKUs that were fragemented. They have now been combine

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

It depends on when you want to consolidate products as the same or not. For example, a different size of bottled water may be better to keep distinguish. There may be more information on what size of water packs people are more likely to buy depending on the season. You would lose this information if the products were grouped.

_your answer here_